# Query 9: Busiest Pickup Areas by Day of Week
**Type:** Grouping by Multiple Attributes
**Problem:** Find the top 5 busiest pickup grid cells for each day of the week.

In [1]:
import time
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName('Q9_BusiestZones') \
    .master('local[*]') \
    .config('spark.sql.shuffle.partitions','8') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

26/04/25 18:27:01 WARN Utils: Your hostname, mariam-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/04/25 18:27:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/25 18:27:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/25 18:27:03 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/25 18:27:03 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/25 18:27:03 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


Spark version: 3.5.1


In [2]:
DATA_PATH = '../data/yellow_tripdata_2015-01.csv'

df = spark.read.option('header','true').option('inferSchema','true').csv(DATA_PATH)

# Create a pickup area identifier from rounded lat/lon
df = df.withColumn('pickup_area',
        F.concat(
            F.round('pickup_latitude',1).cast('string'),
            F.lit('_'),
            F.round('pickup_longitude',1).cast('string')
        )) \
       .withColumn('day_of_week', F.dayofweek('tpep_pickup_datetime')) \
       .withColumn('day_name', F.date_format('tpep_pickup_datetime','EEEE'))

df.createOrReplaceTempView('trips')
rdd = df.rdd
print('Total rows:', df.count())

Total rows: 11157879


## RDD Implementation

In [3]:
from collections import defaultdict
start = time.time()

counts = (
    rdd
    .filter(lambda r: r['day_of_week'] is not None and r['pickup_area'] is not None)
    .map(lambda r: ((int(r['day_of_week']), r['pickup_area']), 1))
    .reduceByKey(lambda a,b: a+b)
    .collect()
)

day_areas = defaultdict(list)
for (day, area), cnt in counts:
    day_areas[day].append((area, cnt))

days = {1:'Sunday',2:'Monday',3:'Tuesday',4:'Wednesday',
        5:'Thursday',6:'Friday',7:'Saturday'}
result_rdd = []
for day in sorted(day_areas):
    for rank,(area,cnt) in enumerate(sorted(day_areas[day], key=lambda x:-x[1])[:5], 1):
        result_rdd.append((days.get(day,str(day)), rank, area, cnt))

rdd_time = time.time() - start
print(f'RDD | Time: {rdd_time:.2f}s')
print(f'{"Day":<12} {"Rank":>5} {"Area":>18} {"Trips":>10}')
for row in result_rdd[:14]:
    print(f'{row[0]:<12} {row[1]:>5} {row[2]:>18} {row[3]:>10,}')

RDD | Time: 102.93s
Day           Rank               Area      Trips
Sunday           1         40.8_-74.0    612,983
Sunday           2         40.7_-74.0    580,591
Sunday           3         40.8_-73.9     76,956
Sunday           4         40.6_-73.8     28,601
Sunday           5            0.0_0.0     24,208
Monday           1         40.8_-74.0    614,110
Monday           2         40.7_-74.0    404,436
Monday           3         40.8_-73.9     69,998
Monday           4         40.6_-73.8     28,991
Monday           5            0.0_0.0     20,825
Tuesday          1         40.8_-74.0    684,315
Tuesday          2         40.7_-74.0    446,226
Tuesday          3         40.8_-73.9     62,579
Tuesday          4            0.0_0.0     23,541


## DataFrame Implementation

In [4]:
start = time.time()

area_day = (
    df.filter(F.col('pickup_area').isNotNull())
      .groupBy('day_name','day_of_week','pickup_area')
      .agg(F.count('*').alias('trip_count'))
)

window_spec = Window.partitionBy('day_of_week').orderBy(F.desc('trip_count'))

result_df = (
    area_day
    .withColumn('rank', F.rank().over(window_spec))
    .filter(F.col('rank') <= 5)
    .orderBy('day_of_week','rank')
    .select('day_name','pickup_area','trip_count','rank')
)
result_df.explain(True)
df_time = time.time() - start
print(f'DataFrame | Time: {df_time:.2f}s')
result_df.show(35)

== Parsed Logical Plan ==
'Project ['day_name, 'pickup_area, 'trip_count, 'rank]
+- Sort [day_of_week#77 ASC NULLS FIRST, rank#198 ASC NULLS FIRST], true
   +- Filter (rank#198 <= 5)
      +- Project [day_name#99, day_of_week#77, pickup_area#55, trip_count#191L, rank#198]
         +- Project [day_name#99, day_of_week#77, pickup_area#55, trip_count#191L, rank#198, rank#198]
            +- Window [rank(trip_count#191L) windowspecdefinition(day_of_week#77, trip_count#191L DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS rank#198], [day_of_week#77], [trip_count#191L DESC NULLS LAST]
               +- Project [day_name#99, day_of_week#77, pickup_area#55, trip_count#191L]
                  +- Aggregate [day_name#99, day_of_week#77, pickup_area#55], [day_name#99, day_of_week#77, pickup_area#55, count(1) AS trip_count#191L]
                     +- Filter isnotnull(pickup_area#55)
                        +- Project [VendorID#17, tpep_pickup_datetime#18, 

+---------+-----------+----------+----+
| day_name|pickup_area|trip_count|rank|
+---------+-----------+----------+----+
|   Sunday| 40.8_-74.0|    612983|   1|
|   Sunday| 40.7_-74.0|    580591|   2|
|   Sunday| 40.8_-73.9|     76956|   3|
|   Sunday| 40.6_-73.8|     28601|   4|
|   Sunday|    0.0_0.0|     24208|   5|
|   Monday| 40.8_-74.0|    614110|   1|
|   Monday| 40.7_-74.0|    404436|   2|
|   Monday| 40.8_-73.9|     69998|   3|
|   Monday| 40.6_-73.8|     28991|   4|
|   Monday|    0.0_0.0|     20825|   5|
|  Tuesday| 40.8_-74.0|    684315|   1|
|  Tuesday| 40.7_-74.0|    446226|   2|
|  Tuesday| 40.8_-73.9|     62579|   3|
|  Tuesday|    0.0_0.0|     23541|   4|
|  Tuesday| 40.6_-73.8|     22411|   5|
|Wednesday| 40.8_-74.0|    815464|   1|
|Wednesday| 40.7_-74.0|    550785|   2|
|Wednesday| 40.8_-73.9|     74908|   3|
|Wednesday|    0.0_0.0|     29613|   4|
|Wednesday| 40.6_-73.8|     25381|   5|
| Thursday| 40.8_-74.0|    993716|   1|
| Thursday| 40.7_-74.0|    728996|   2|


## Spark SQL Implementation

In [5]:
start = time.time()

result_sql = spark.sql("""
    WITH area_day AS (
        SELECT DAYOFWEEK(tpep_pickup_datetime)             AS day_of_week,
               DATE_FORMAT(tpep_pickup_datetime,'EEEE')   AS day_name,
               CONCAT(ROUND(pickup_latitude,1),'_',
                      ROUND(pickup_longitude,1))          AS pickup_area,
               COUNT(*)                                   AS trip_count
        FROM   trips
        WHERE  tpep_pickup_datetime IS NOT NULL
        GROUP BY 1,2,3
    ),
    ranked AS (
        SELECT *,
               RANK() OVER (PARTITION BY day_of_week ORDER BY trip_count DESC) AS zone_rank
        FROM   area_day
    )
    SELECT day_name, pickup_area, trip_count, zone_rank
    FROM   ranked
    WHERE  zone_rank <= 5
    ORDER BY day_of_week, zone_rank
""")
result_sql.explain(True)
sql_time = time.time() - start
print(f'SQL | Time: {sql_time:.2f}s')
result_sql.show(35)

== Parsed Logical Plan ==
CTE [area_day, ranked]
:  :- 'SubqueryAlias area_day
:  :  +- 'Aggregate [1, 2, 3], ['DAYOFWEEK('tpep_pickup_datetime) AS day_of_week#254, 'DATE_FORMAT('tpep_pickup_datetime, EEEE) AS day_name#255, 'CONCAT('ROUND('pickup_latitude, 1), _, 'ROUND('pickup_longitude, 1)) AS pickup_area#256, 'COUNT(1) AS trip_count#257]
:  :     +- 'Filter isnotnull('tpep_pickup_datetime)
:  :        +- 'UnresolvedRelation [trips], [], false
:  +- 'SubqueryAlias ranked
:     +- 'Project [*, 'RANK() windowspecdefinition('day_of_week, 'trip_count DESC NULLS LAST, unspecifiedframe$()) AS zone_rank#258]
:        +- 'UnresolvedRelation [area_day], [], false
+- 'Sort ['day_of_week ASC NULLS FIRST, 'zone_rank ASC NULLS FIRST], true
   +- 'Project ['day_name, 'pickup_area, 'trip_count, 'zone_rank]
      +- 'Filter ('zone_rank <= 5)
         +- 'UnresolvedRelation [ranked], [], false

== Analyzed Logical Plan ==
day_name: string, pickup_area: string, trip_count: bigint, zone_rank: int
WithC

+---------+-----------+----------+---------+
| day_name|pickup_area|trip_count|zone_rank|
+---------+-----------+----------+---------+
|   Sunday| 40.8_-74.0|    612983|        1|
|   Sunday| 40.7_-74.0|    580591|        2|
|   Sunday| 40.8_-73.9|     76956|        3|
|   Sunday| 40.6_-73.8|     28601|        4|
|   Sunday|    0.0_0.0|     24208|        5|
|   Monday| 40.8_-74.0|    614110|        1|
|   Monday| 40.7_-74.0|    404436|        2|
|   Monday| 40.8_-73.9|     69998|        3|
|   Monday| 40.6_-73.8|     28991|        4|
|   Monday|    0.0_0.0|     20825|        5|
|  Tuesday| 40.8_-74.0|    684315|        1|
|  Tuesday| 40.7_-74.0|    446226|        2|
|  Tuesday| 40.8_-73.9|     62579|        3|
|  Tuesday|    0.0_0.0|     23541|        4|
|  Tuesday| 40.6_-73.8|     22411|        5|
|Wednesday| 40.8_-74.0|    815464|        1|
|Wednesday| 40.7_-74.0|    550785|        2|
|Wednesday| 40.8_-73.9|     74908|        3|
|Wednesday|    0.0_0.0|     29613|        4|
|Wednesday

## Performance Comparison

In [6]:
print('='*65)
print(f'{"Metric":<25} {"RDD":>12} {"DataFrame":>12} {"SQL":>12}')
print('-'*65)
print(f'{"Execution Time":<25} {rdd_time:>11.2f}s {df_time:>11.2f}s {sql_time:>11.2f}s')
print(f'{"Top-N per group":<25} {"Manual":>12} {"RANK()":>12} {"RANK()":>12}')
print(f'{"Grouping Keys":<25} {"(day,area)":>12} {"(day,area)":>12} {"(day,area)":>12}')
print('='*65)
print('KEY INSIGHT: Catalyst pushes RANK window into a single pass.')
print('RDD requires manual collect + Python sort = driver bottleneck.')

Metric                             RDD    DataFrame          SQL
-----------------------------------------------------------------
Execution Time                 102.93s        0.49s        0.37s
Top-N per group                 Manual       RANK()       RANK()
Grouping Keys               (day,area)   (day,area)   (day,area)
KEY INSIGHT: Catalyst pushes RANK window into a single pass.
RDD requires manual collect + Python sort = driver bottleneck.
